#### Dataset

We use the Digits dataset provided by scikit-learn.

- 1,797 handwritten digit images
- Image size: 8 × 8 pixels
- 64 input features per image
- 10 target classes (digits 0–9)

<h1> End-to-End Machine Learning Pipeline

We will start by building a simple, functional end-to-end ML pipeline. We won’t worry about Docker or CI/CD for now; first, we need a model where the training, evaluation, and saving processes work correctly. Later, we will transform this code into a real-world service.

### Objective

Build a reproducible machine learning pipeline that covers data preparation, model training, evaluation, and model persistence.

### Project

Handwritten Digit Classification using the scikit-learn Digits dataset.

### Pipeline

1. Load and inspect the dataset.
2. Split data into training and test sets.
3. Build a preprocessing and classification pipeline.
4. Train and evaluate the model.
5. Save the trained pipeline.
6. Load the saved model and perform inference.
7. Test and deploy the model using FastAPI and Docker.
8. Automate testing and image publishing with GitHub Actions.

### Engineering Principles

- Reproducibility
- Separation of training and inference
- Data leakage prevention
- Automated testing
- Containerization
- Continuous Integration and Delivery

In [5]:
import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from pathlib import Path
import joblib


<h3> Step 1: Loading and Splitting Data

In [2]:
# Load dataset
digits = load_digits()

X = digits.data
y = digits.target

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))

print("\nTraining samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

print("Input features:", X_train.shape[1])

Dataset shape: (1797, 64)
Number of classes: 10

Training samples: 1437
Test samples: 360
Input features: 64


<h3> Step 2: Building the First Pipeline

Unlike the previous section, where we implemented a neural network from scratch, this time we will use ready-made tools from `scikit-learn`. The focus here is on model engineering, rather than re-implementing algorithms.

We will utilize two components:
- `StandardScaler`: For feature standardization.
- `LogisticRegression`: To build a baseline model for ten-class classification.

By combining both into a `Pipeline`, the preprocessing steps and the model are encapsulated and executed as a single, unified unit.

Engineering Note: Since the `StandardScaler` is included within a `Pipeline`, it is fitted only on the training data, and the same transformation is then applied to the test data. This prevents information from the test set from leaking into the training process.

This code is a Logistic Regression model placed inside a pipeline.

A pipeline means connecting several processing stages in sequence so that data passes through them.

StandardScaler learns the mean and standard deviation from the training data and uses them to transform the data.

Parameter `max_iter=2000`:

This means the model's optimization algorithm is permitted to perform a maximum of 2,000 iterations to determine the model parameters.

This figure does not represent 2,000 full training passes over the entire dataset; rather, it serves as a limit on the number of iterations for the optimization algorithm.

If this value is set too low, the model may fail to converge. Increasing it can provide more opportunity for convergence, though it does not guarantee that the model will necessarily converge.

Preventing Data Leakage: We should not fit the scaler separately on the entire dataset—including the test data—because the statistical information from the test data would then leak into the training process.

In [4]:
# Build pipeline
model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Train
model_pipeline.fit(X_train, y_train)

# Evaluate
train_predictions = model_pipeline.predict(X_train)
test_predictions = model_pipeline.predict(X_test)

train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_test, test_predictions))

Training accuracy: 0.9993
Test accuracy: 0.9722

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       0.91      0.89      0.90        36
           2       1.00      1.00      1.00        35
           3       0.97      1.00      0.99        37
           4       0.95      1.00      0.97        36
           5       1.00      1.00      1.00        37
           6       1.00      0.97      0.99        36
           7       1.00      0.97      0.99        36
           8       0.89      0.91      0.90        35
           9       1.00      0.97      0.99        36

    accuracy                           0.97       360
   macro avg       0.97      0.97      0.97       360
weighted avg       0.97      0.97      0.97       360



<h3> Step 3: Model Persistence (Saving and Loading Models)

Model persistence refers to saving a trained model so that it can be reloaded and used later without the need for retraining.

In our project, the following elements must be preserved:
- The learned parameters of the `LogisticRegression` model.
- The parameters of the `StandardScaler` calculated during training.
- The structure of the `Pipeline` and the sequence of its constituent steps.

It is important to note that we do not simply have a classification model; rather, we have a complete pipeline that first standardizes the data and subsequently performs predictions.

For storage, we use the `joblib` library, which is suitable for saving and loading Python objects, including scikit-learn models.

In [6]:
# Create the artifacts directory
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Define the model file path
model_path = ARTIFACTS_DIR / "digits_pipeline.joblib"

# Save the complete trained pipeline
joblib.dump(model_pipeline, model_path)

print("Model saved:", model_path)
print("File exists:", model_path.exists())
print("File size (KB):", round(model_path.stat().st_size / 1024, 2))

Model saved: artifacts\digits_pipeline.joblib
File exists: True
File size (KB): 7.88


<h3> Step 4: Model Reloading and Validation

Saving the file alone is not enough. We must ensure that the model can be loaded and that it yields the same predictions as before.

In [7]:
# Load the saved pipeline
loaded_pipeline = joblib.load(model_path)

# Predict using the loaded model
reloaded_predictions = loaded_pipeline.predict(X_test)

# Verify predictions and evaluate performance
predictions_identical = np.array_equal(
    test_predictions,
    reloaded_predictions
)

reloaded_accuracy = accuracy_score(
    y_test,
    reloaded_predictions
)

print("Predictions identical:", predictions_identical)
print(f"Reloaded test accuracy: {reloaded_accuracy:.4f}")

Predictions identical: True
Reloaded test accuracy: 0.9722


This experiment is a simple example of post-storage validation—a practice that is also important in real-world projects.

<h3> Step 5: First Independent Inference

In [8]:
# Select one unseen test sample
sample_index = 0

sample = X_test[sample_index].reshape(1, -1)
true_label = y_test[sample_index]

# Predict using the loaded pipeline
predicted_label = loaded_pipeline.predict(sample)[0]

# Get class probabilities
probabilities = loaded_pipeline.predict_proba(sample)[0]
confidence = probabilities.max()

print("True digit:", true_label)
print("Predicted digit:", predicted_label)
print(f"Prediction confidence: {confidence:.4f}")
print("Input shape:", sample.shape)

True digit: 5
Predicted digit: 5
Prediction confidence: 0.9171
Input shape: (1, 64)


In this example, "confidence" is the highest probability predicted by the model; this value does not necessarily represent a calibrated confidence or the true probability that the prediction is correct.

Up to this point, we have done everything within a Notebook. However, in a real-world project, we do not want to execute Notebook cells every time we need to perform training or make predictions.

Instead, we want to organize the code into independent components that can be executed via the terminal, an API, automated tests, or—in the future—Docker.

In our project, we separate responsibilities:
- train.py: Receives data, trains the model, evaluates it, and saves the output files.
- predict.py: Loads the saved model and generates predictions for new input.
- 01_end_to_end_ml_pipeline.ipynb: Serves as our training and experimentation environment.

This separation is important, as the final service should not have to retrain the model from scratch every time it receives a request.

<h3> Step 6: Prediction Checking

In [9]:
from src.predict import load_model, predict_digit

loaded_model = load_model()

result = predict_digit(X_test[0], loaded_model)

print("True digit:", y_test[0])
print("Prediction:", result)

True digit: 5
Prediction: {'predicted_digit': 5, 'confidence': 0.9171498136833252}


<h3> Step 7: Automated testing with pytest